# IEEE 9-bus with mixed synchronous, grid-forming and grid-following sources

IEEE 9-bus in EMT, power-flow initialized. gen1 is a 4th-order synchronous machine, gen2 a grid-forming SSN inverter, gen3 a grid-following averaged VSI. The notebook runs the compiled example and plots the shared behaviour. The grid-forming tuning is exposed through `gfm_*` options and can be swept.

In [ ]:
from villas.dataprocessing.readtools import read_timeseries_csv
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
import subprocess

duration = 2.0

root_path = (
    subprocess.Popen(["git", "rev-parse", "--show-toplevel"], stdout=subprocess.PIPE)
    .communicate()[0]
    .rstrip()
    .decode("utf-8")
)

name = "EMT_Ph3_IEEE9_SSN_InverterMix"
candidates = [os.path.join(root_path, "build", "dpsim", "examples", "cxx")]
candidates += glob.glob(
    os.path.join(root_path, "build", "*", "dpsim", "examples", "cxx")
)
path_exec = next(p for p in candidates if os.path.isfile(os.path.join(p, name)))
print("binary:", path_exec)

In [ ]:
# Run the example, optionally overriding the grid-forming tuning, and return
# the logged time series (the single trailing preallocated sample is dropped).
def run(**gfm_options):
    cmd = [os.path.join(path_exec, name), "-d", str(duration), "--option", "log=true"]
    for key, value in gfm_options.items():
        cmd += ["--option", f"{key}={value}"]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)
    return read_timeseries_csv("logs/" + name + "/" + name + ".csv")


def xy(ts, key):
    m = ts[key].time > 1e-9
    return ts[key].time[m], ts[key].values[m]


# For a balanced set sqrt(a^2 + b^2 + c^2) of the phase voltages equals the
# line-to-line RMS magnitude.
def bus_ll_rms(ts, bus):
    t, a = xy(ts, bus + "_0")
    _, b = xy(ts, bus + "_1")
    _, c = xy(ts, bus + "_2")
    return t, np.sqrt(a**2 + b**2 + c**2)


ts = run()

Bus voltages at the grid-forming (BUS2) and grid-following (BUS3) points of connection, and the anchoring machine bus (BUS1).

In [ ]:
plt.figure(figsize=(12, 6))
for bus in ["BUS1", "BUS2", "BUS3"]:
    t, v = bus_ll_rms(ts, bus)
    plt.plot(t, v / 1e3, label=bus)
plt.xlabel("time [s]")
plt.ylabel("|V| line-to-line RMS [kV]")
plt.legend()
plt.show()

Active and reactive power of the two inverters. The grid-forming gen2 holds its dispatch while forming the voltage; the grid-following gen3 tracks its power reference.

In [ ]:
fig, (axp, axq) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for gen in ["GEN2", "GEN3"]:
    t, p = xy(ts, gen + ".p_inst")
    _, q = xy(ts, gen + ".q_inst")
    axp.plot(t, p / 1e6, label=gen)
    axq.plot(t, q / 1e6, label=gen)
axp.set_ylabel("P [MW]")
axq.set_ylabel("Q [MVAr]")
axq.set_xlabel("time [s]")
axp.legend()
axq.legend()
plt.show()

Grid-forming frequency (gen2) against the synchronous machine speed (gen1), both in Hz. They stay synchronised.

In [ ]:
plt.figure(figsize=(12, 6))
t, w_gfm = xy(ts, "GEN2.omega")
plt.plot(t, w_gfm / (2 * np.pi), label="GEN2 grid-forming")
t, w_sg = xy(ts, "GEN1.omega")
plt.plot(t, w_sg * 60.0, label="GEN1 synchronous")
plt.xlabel("time [s]")
plt.ylabel("frequency [Hz]")
plt.legend()
plt.show()

Sweeping the tuning. On a stiff grid the grid-forming source needs enough damping; here the swing-damping `gfm_d` is swept and the gen2 active power is compared.

Grid-forming knobs: `gfm_d`, `gfm_dq`, `gfm_dqc`, `gfm_kpv`, `gfm_kiv`, `gfm_ff`, `gfm_rv`. Grid-following knobs (gen3): `gfl_kppll`, `gfl_kipll`, `gfl_kpp`, `gfl_kip`, `gfl_kpi`, `gfl_kii`. Pass any of them to `run(...)`, e.g. `run(gfl_kppll=0.02)`.

In [ ]:
plt.figure(figsize=(12, 6))
for damping in [3.0e5, 1.0e6, 3.0e6]:
    ts_d = run(gfm_d=damping)
    t, p = xy(ts_d, "GEN2.p_inst")
    plt.plot(t, p / 1e6, label=f"gfm_d = {damping:.0e}")
plt.xlabel("time [s]")
plt.ylabel("GEN2 P [MW]")
plt.legend()
plt.show()